# Advanced tutorial problems: `__slots__` with single inheritance
## A second, step-by-step problem collection — complete solutions and runnable experiments

**Level:** intermediate to advanced · **Python:** 3.10+ · **Dependencies:** standard library only

This is a **different collection** from the earlier 24-problem book. It follows the source lesson's teaching rhythm: introduce one question, examine a small class hierarchy, predict an outcome, run short cells, inspect the result, and explain *why*. It extends the original lesson with additional behavior involving slot declaration, introspection, descriptor access, validation, caching, pattern matching, class construction, and API design. Those extensions are new exercises, not claims that they appeared in the supplied notes.

**How to work through it:** For each problem, stop at the “Before running anything” question and write down a prediction. The solution is intentionally revealed gradually over several code and markdown cells. Finish with the assertions and optional variant. Use **Kernel → Restart & Run All** for a reproducibility check.

**Scope:** one inheritance chain at a time (no multiple-inheritance layout conflicts). Slot behavior is described for ordinary Python classes; any implementation-dependent behavior is identified. A slot constrains available attribute *names* in a dictionary-free instance; it does not enforce types, immutability, or security.

## Lab setup — small helpers for precise experiments

The helpers keep each demonstration short. We check exception **types**, not platform-dependent exception messages. Every problem uses uniquely named classes to avoid notebook redefinition surprises.

In [1]:
import sys
from functools import cached_property

assert sys.version_info >= (3, 10), 'Python 3.10 or later is required.'


def raises(exc_type, function, /, *args, **kwargs):
    """Return the expected exception; fail clearly if the call behaves differently."""
    try:
        function(*args, **kwargs)
    except exc_type as exc:
        return exc
    except Exception as exc:
        raise AssertionError(
            f'Expected {exc_type.__name__}, got {type(exc).__name__}'
        ) from exc
    raise AssertionError(f'Expected {exc_type.__name__}, but nothing was raised')


MISSING = object()  # A sentinel distinct from None and any valid cached value.
print('Python', sys.version.split()[0], '— tutorial helpers ready')

Python 3.13.7 — tutorial helpers ready


## Problem 01 — The single-name slot declaration trap

**Challenge.** Determine whether the strings `"token"` and `( "token", )` produce the same slot, then fix a class with an accidentally misspelled slot.

**Before running anything:** If a slot declaration is one string, does Python iterate over its letters? What differs between a string and a one-element tuple?

The next cells form a guided, fully worked solution. Read the explanation, make your prediction, then run the next cell.

**Solution · Step 1.** Define a one-name slot using both supported spellings. Compare their class dictionaries, which hold generated slot descriptors.

In [2]:
class T01String:
    __slots__ = 'token'


class T01Tuple:
    __slots__ = ('token',)


print('String declaration:', T01String.__slots__)
print('Tuple declaration: ', T01Tuple.__slots__)
print('Generated descriptor names:', [k for k in T01String.__dict__ if k == 'token'])

String declaration: token
Tuple declaration:  ('token',)
Generated descriptor names: ['token']


**What just happened?** A string is treated as **one slot name**, not six character-sized slots. The trailing comma in `("token",)` makes a one-item tuple; without it, parentheses merely group a string.

**Solution · Step 2.** Now introduce an actual error: declare `"tokne"`, but assign to `token` during initialization. Trap the failure before fixing it.

In [3]:
class T01Broken:
    __slots__ = ('tokne',)

    def __init__(self, token):
        self.token = token


raises(AttributeError, T01Broken, 'A-1')
print('The initializer requested a name that has no slot.')

The initializer requested a name that has no slot.


**Solution · Step 3.** Apply the fix by creating a new class with the correct slot name, not by trying to mutate the old class layout.

In [4]:
class T01Fixed:
    __slots__ = ('token',)

    def __init__(self, token):
        self.token = token


fixed_01 = T01Fixed('A-1')
print(fixed_01.token)

A-1


**Verification.** Confirm that the two valid declarations behave alike, whereas unknown instance fields are rejected. These checks should pass when you run the notebook from top to bottom.

In [5]:
assert T01String.__slots__ == 'token'
assert T01Tuple.__slots__ == ('token',)
assert fixed_01.token == 'A-1'
assert not hasattr(fixed_01, '__dict__')
raises(AttributeError, setattr, fixed_01, 'tokne', 'incorrect')

AttributeError("'T01Fixed' object has no attribute 'tokne' and no __dict__ for setting new attributes")

**Takeaway.** `__slots__ = "name"` and `__slots__ = ("name",)` declare the same single attribute. A typo in a slot name usually surfaces as an `AttributeError` when assigning the intended name.

**Try it yourself.** Change the broken class to `__slots__ = ("tokne", "token")` and explain why this is not as clean as correcting the typo.

## Problem 02 — Iterables used as slot definitions

**Challenge.** Build a class using a generator as its slot declaration. Explain why introspecting its `__slots__` afterward can be misleading, then redesign it with a tuple.

**Before running anything:** Does inspecting `Class.__slots__` necessarily reveal all slot names if the declaration used a one-pass iterator?

The next cells form a guided, fully worked solution. Read the explanation, make your prediction, then run the next cell.

**Solution · Step 1.** Supply a generator expression to the class statement and inspect both the original declaration and created descriptors.

In [6]:
class T02OnePass:
    __slots__ = (name for name in ('left', 'right'))


print('Stored declaration:', list(T02OnePass.__slots__))
print('Actual descriptors:',
      [name for name in ('left', 'right') if name in T02OnePass.__dict__])

Stored declaration: []
Actual descriptors: ['left', 'right']


**What just happened?** Python consumes the iterator when creating the slot descriptors. The iterator retained in `__slots__` has therefore been exhausted. This is legal, but a poor choice for readable introspection.

**Solution · Step 2.** Create an instance to prove that the descriptors exist despite the empty iterator reported above.

In [7]:
one_pass_02 = T02OnePass()
one_pass_02.left = 7
one_pass_02.right = 9
print(one_pass_02.left, one_pass_02.right)
raises(AttributeError, setattr, one_pass_02, 'center', 8)

7 9


AttributeError("'T02OnePass' object has no attribute 'center' and no __dict__ for setting new attributes")

**Solution · Step 3.** Prefer an immutable, reusable declaration. Distinguish *declared names* from attributes that have received a value.

In [8]:
class T02Stable:
    __slots__ = ('left', 'right')


stable_02 = T02Stable()
stable_02.left = 7
print('Declared:', T02Stable.__slots__)
print('Initialized:', [(name, getattr(stable_02, name))
                       for name in T02Stable.__slots__
                       if hasattr(stable_02, name)])

Declared: ('left', 'right')
Initialized: [('left', 7)]


**Verification.** Prove the generator class really has both slots and that the tuple remains inspectable. These checks should pass when you run the notebook from top to bottom.

In [9]:
assert (one_pass_02.left, one_pass_02.right) == (7, 9)
assert list(T02OnePass.__slots__) == []
assert T02Stable.__slots__ == ('left', 'right')
assert hasattr(stable_02, 'left') and not hasattr(stable_02, 'right')

**Takeaway.** Define `__slots__` with an explicit tuple (or another stable iterable). A generator can create the intended layout while leaving misleading introspection behind.

**Try it yourself.** Try `__slots__ = ["left", "right"]`. How does its readability compare with a generator?

## Problem 03 — Declared is not the same as initialized

**Challenge.** Design a slotted base and child whose optional fields start unset. Write a safe snapshot that distinguishes an absent field from an assigned `None`.

**Before running anything:** Will `dir(instance)` omit a declared but unset slot? Will `vars(instance)` work?

The next cells form a guided, fully worked solution. Read the explanation, make your prediction, then run the next cell.

**Solution · Step 1.** Create an intentionally partially initialized single-inheritance chain.

In [10]:
class T03Packet:
    __slots__ = ('packet_id', 'payload')

    def __init__(self, packet_id):
        self.packet_id = packet_id


class T03SecurePacket(T03Packet):
    __slots__ = ('signature',)


packet_03 = T03SecurePacket('PKT-1')
print('payload visible in dir:', 'payload' in dir(packet_03))
print('signature visible in dir:', 'signature' in dir(packet_03))

payload visible in dir: True
signature visible in dir: True


**What just happened?** `dir` reports discoverable *names*, not whether a slot currently holds a value. Reading an uninitialized slot raises `AttributeError`; the instance also has no `__dict__`, so `vars` is not a universal serializer.

**Solution · Step 2.** Demonstrate both failure modes explicitly.

In [11]:
raises(AttributeError, getattr, packet_03, 'payload')
raises(AttributeError, getattr, packet_03, 'signature')
raises(TypeError, vars, packet_03)
print('Three expected exceptions were handled.')

Three expected exceptions were handled.


**Solution · Step 3.** Use a unique sentinel to preserve the difference between unset and `None`.

In [12]:
def snapshot_03(obj, fields):
    result = {}
    for name in fields:
        value = getattr(obj, name, MISSING)
        result[name] = '<UNSET>' if value is MISSING else value
    return result


fields_03 = ('packet_id', 'payload', 'signature')
print(snapshot_03(packet_03, fields_03))
packet_03.payload = None
packet_03.signature = b'signed'
print(snapshot_03(packet_03, fields_03))

{'packet_id': 'PKT-1', 'payload': '<UNSET>', 'signature': '<UNSET>'}
{'packet_id': 'PKT-1', 'payload': None, 'signature': b'signed'}


**Verification.** Check that explicit `None` survives, rather than being misreported as absent. These checks should pass when you run the notebook from top to bottom.

In [13]:
assert snapshot_03(packet_03, fields_03) == {
    'packet_id': 'PKT-1', 'payload': None, 'signature': b'signed'
}
assert not hasattr(packet_03, '__dict__')

**Takeaway.** `__slots__` declares possible storage; it does not initialize it. Use `getattr(obj, name, sentinel)` when an unset slot must be distinguishable from a valid `None`.

**Try it yourself.** Delete `signature`, rerun the snapshot, and explain the change.

## Problem 04 — A descriptor survives even when the attribute is hidden

**Challenge.** Investigate what happens when a class attribute replaces a slot descriptor at runtime. Recover the original descriptor and restore normal access.

**Before running anything:** If the descriptor is replaced in the class dictionary, where does the existing slot value go?

The next cells form a guided, fully worked solution. Read the explanation, make your prediction, then run the next cell.

**Solution · Step 1.** Inspect and save the descriptor before changing anything.

In [14]:
class T04Record:
    __slots__ = ('code',)

    def __init__(self, code):
        self.code = code


record_04 = T04Record('ORIGINAL')
saved_descriptor_04 = T04Record.__dict__['code']
print(type(saved_descriptor_04).__name__)
print(saved_descriptor_04.__get__(record_04, T04Record))

member_descriptor
ORIGINAL


**What just happened?** For ordinary Python classes, slot declarations create descriptor objects in the class dictionary. The descriptor controls access to the value stored in each instance.

**Solution · Step 2.** Replace the class descriptor temporarily. Inspect attribute lookup and attempt an ordinary assignment.

In [15]:
T04Record.code = 'CLASS FALLBACK'
try:
    print('Normal lookup:', record_04.code)
    raises(AttributeError, setattr, record_04, 'code', 'NEW')
    print('Descriptor still reads storage:',
          saved_descriptor_04.__get__(record_04, T04Record))
finally:
    T04Record.code = saved_descriptor_04

Normal lookup: CLASS FALLBACK
Descriptor still reads storage: ORIGINAL


**What just happened?** Overwriting the class attribute hides the descriptor from ordinary lookup; it does **not** erase existing slot storage. Since the descriptor was restored in `finally`, subsequent notebook cells are safe even if an assertion fails. Replacing descriptors like this is a demonstration of a pitfall, not a recommended application technique.

**Solution · Step 3.** Verify that normal slot lookup is restored and the original value was retained.

In [16]:
print('Restored:', record_04.code)
record_04.code = 'UPDATED'
print('Updated via restored descriptor:', record_04.code)

Restored: ORIGINAL
Updated via restored descriptor: UPDATED


**Verification.** Check both restoration and the absence of a dictionary. These checks should pass when you run the notebook from top to bottom.

In [17]:
assert T04Record.__dict__['code'] is saved_descriptor_04
assert record_04.code == 'UPDATED'
assert not hasattr(record_04, '__dict__')

**Takeaway.** The class-level slot descriptor is essential to normal access. Do not overwrite it with a same-named class value.

**Try it yourself.** Why is a regular `DEFAULT_CODE` class constant safer than assigning `code = "DEFAULT"` after class creation?

## Problem 05 — Calling an inherited slot descriptor directly

**Challenge.** Build a slotted parent and child, then demonstrate how a descriptor accesses an inherited slot independently of a convenience method.

**Before running anything:** Where is the descriptor for an inherited field located: in the child `__dict__`, the base `__dict__`, or both?

The next cells form a guided, fully worked solution. Read the explanation, make your prediction, then run the next cell.

**Solution · Step 1.** Declare two classes and inspect only their own class dictionaries.

In [18]:
class T05Base:
    __slots__ = ('serial',)

    def __init__(self, serial):
        self.serial = serial


class T05Child(T05Base):
    __slots__ = ('batch',)

    def __init__(self, serial, batch):
        super().__init__(serial)
        self.batch = batch


part_05 = T05Child('S-20', 'B-3')
print('Base owns serial:', 'serial' in T05Base.__dict__)
print('Child owns serial:', 'serial' in T05Child.__dict__)
print('Child owns batch:', 'batch' in T05Child.__dict__)

Base owns serial: True
Child owns serial: False
Child owns batch: True


**What just happened?** Inherited slots are not automatically copied into the subclass’s class dictionary. Python discovers the base descriptor through the method resolution order.

**Solution · Step 2.** Retrieve that one descriptor and call its read and write methods explicitly.

In [19]:
serial_descriptor_05 = T05Base.__dict__['serial']
print(serial_descriptor_05.__get__(part_05, T05Child))
serial_descriptor_05.__set__(part_05, 'S-21')
print('Ordinary lookup after descriptor write:', part_05.serial)

S-20
Ordinary lookup after descriptor write: S-21


**Solution · Step 3.** Confirm which attributes are initialized and which class owns each descriptor.

In [20]:
for owner, name in ((T05Base, 'serial'), (T05Child, 'batch')):
    descriptor = vars(owner)[name]  # vars(class) is valid; vars(slotted instance) is not.
    print(owner.__name__, name,
          descriptor.__get__(part_05, T05Child))

T05Base serial S-21
T05Child batch B-3


**Verification.** Check inherited storage and the correct descriptor locations. These checks should pass when you run the notebook from top to bottom.

In [21]:
assert (part_05.serial, part_05.batch) == ('S-21', 'B-3')
assert 'serial' not in T05Child.__dict__
assert 'batch' in T05Child.__dict__
raises(TypeError, vars, part_05)

TypeError('vars() argument must have __dict__ attribute')

**Takeaway.** To inspect declared slots, check the class defining them (or walk the MRO). A child’s `__slots__` lists its new slots, not its total inherited storage.

**Try it yourself.** Write a loop over `T05Child.__mro__` that prints only classes declaring `__slots__`.

## Problem 06 — Trace reads without infinite recursion

**Challenge.** Instrument attribute reads on a slotted base while retaining a slotted child. Implement `__getattribute__` without recursively calling itself.

**Before running anything:** What happens if the implementation uses `self.log` to record every read, including reads of `log`?

The next cells form a guided, fully worked solution. Read the explanation, make your prediction, then run the next cell.

**Solution · Step 1.** Implement a safe override using `object.__getattribute__` and a class-level log for this small demonstration.

In [22]:
class T06Audited:
    __slots__ = ('name',)
    read_log = []

    def __init__(self, name):
        self.name = name

    def __getattribute__(self, name):
        # No `self.name` or `self.read_log` inside this hook.
        if not name.startswith('__'):
            type(self).read_log.append(name)
        return object.__getattribute__(self, name)


class T06Child(T06Audited):
    __slots__ = ('level',)

    def __init__(self, name, level):
        super().__init__(name)
        self.level = level

**What just happened?** `object.__getattribute__` performs the actual lookup while avoiding a second call through the overriding method. `type(self).read_log` retrieves the log through the class rather than through `self`, avoiding a reentrant lookup of `self.read_log`. A class-level log is shared by instances and is used here only for an observable teaching example.

**Solution · Step 2.** Exercise both inherited and local slots and read an invalid field.

In [23]:
T06Audited.read_log.clear()
audit_06 = T06Child('Mina', 4)
print(audit_06.name, audit_06.level)
raises(AttributeError, getattr, audit_06, 'unknown')
print('Read names:', T06Audited.read_log)

Mina 4
Read names: ['name', 'level', 'unknown']


**Solution · Step 3.** Explain why this override changes behavior but does not alter storage.

In [24]:
print('Dictionary available?', hasattr(audit_06, '__dict__'))
print('Own child slots:', T06Child.__slots__)
print('Inherited name:', audit_06.name)

Dictionary available? False
Own child slots: ('level',)
Inherited name: Mina


**Verification.** Confirm that the audit intercepted both successful and failed attribute lookups. These checks should pass when you run the notebook from top to bottom.

In [25]:
assert 'name' in T06Audited.read_log
assert 'level' in T06Audited.read_log
assert 'unknown' in T06Audited.read_log
assert not hasattr(audit_06, '__dict__')

**Takeaway.** Attribute-access hooks and slot layout solve different problems. Use `object.__getattribute__` for actual lookup inside an override; otherwise recursion can occur.

**Try it yourself.** Could a global class-level log be appropriate for independent users? Replace it with a per-instance list and a corresponding `_read_log` slot if you need isolation.

## Problem 07 — Validation in `__setattr__` across one inheritance chain

**Challenge.** Allow only nonnegative integer `quantity` values in the base class and nonempty string `label` values in its child. Preserve normal slot behavior for other attributes.

**Before running anything:** Does declaring an integer-like slot automatically reject negative numbers or strings?

The next cells form a guided, fully worked solution. Read the explanation, make your prediction, then run the next cell.

**Solution · Step 1.** Write the parent’s validation once and delegate storage to `object.__setattr__`.

In [26]:
class T07Stock:
    __slots__ = ('quantity',)

    def __init__(self, quantity):
        self.quantity = quantity

    def __setattr__(self, name, value):
        if name == 'quantity' and (type(value) is not int or value < 0):
            raise ValueError('quantity must be a nonnegative integer')
        object.__setattr__(self, name, value)

**What just happened?** Slots themselves do not validate values. `type(value) is int` intentionally excludes `bool`, which is a subclass of `int`. Delegating to `object.__setattr__` preserves Python’s normal slot checks.

**Solution · Step 2.** Extend validation in the child, then delegate all other assignments through `super()`.

In [27]:
class T07LabeledStock(T07Stock):
    __slots__ = ('label',)

    def __init__(self, quantity, label):
        super().__init__(quantity)
        self.label = label

    def __setattr__(self, name, value):
        if name == 'label' and (not isinstance(value, str) or not value.strip()):
            raise ValueError('label must be nonempty text')
        super().__setattr__(name, value)


stock_07 = T07LabeledStock(8, 'widgets')
print(stock_07.quantity, stock_07.label)

8 widgets


**Solution · Step 3.** Probe every validation boundary, including unrelated names.

In [28]:
raises(ValueError, setattr, stock_07, 'quantity', -2)
raises(ValueError, setattr, stock_07, 'quantity', True)
raises(ValueError, setattr, stock_07, 'label', '  ')
raises(AttributeError, setattr, stock_07, 'color', 'blue')
stock_07.quantity = 0
print('Zero is valid:', stock_07.quantity)

Zero is valid: 0


**Verification.** Ensure valid values survive rejected assignments and both classes remain dictionary-free. These checks should pass when you run the notebook from top to bottom.

In [29]:
assert (stock_07.quantity, stock_07.label) == (0, 'widgets')
assert not hasattr(stock_07, '__dict__')
assert T07LabeledStock.__slots__ == ('label',)

**Takeaway.** Put each class’s validation at its own boundary; delegate to `super().__setattr__` so the full single-inheritance chain participates. This is API validation, not a security boundary: low-level descriptor calls can bypass custom `__setattr__`.

**Try it yourself.** Test whether `object.__setattr__(stock_07, "quantity", -1)` uses your custom validation. Restore a valid value afterward.

## Problem 08 — Extend a property setter without making a second backing slot

**Challenge.** Create a base `temperature` property backed by `_temperature`. Extend the setter in the subclass to reject values above a device-specific maximum, while keeping only one storage slot.

**Before running anything:** Should the child declare `temperature`, `_temperature`, or neither in its own `__slots__`?

The next cells form a guided, fully worked solution. Read the explanation, make your prediction, then run the next cell.

**Solution · Step 1.** Declare the property in the parent and store only its backing field in a slot.

In [30]:
class T08Sensor:
    __slots__ = ('_temperature',)

    def __init__(self, temperature):
        self.temperature = temperature

    @property
    def temperature(self):
        return self._temperature

    @temperature.setter
    def temperature(self, value):
        if not isinstance(value, (int, float)) or isinstance(value, bool):
            raise TypeError('temperature must be numeric')
        self._temperature = float(value)

**What just happened?** The **property name** is a class-level descriptor, not an instance storage slot. Only `_temperature` needs storage. Repeating `_temperature` in a child would hide its inherited descriptor, so do not do that.

**Solution · Step 2.** Use the parent property’s setter as a reusable validation step before applying the child’s extra bound.

In [31]:
class T08LimitedSensor(T08Sensor):
    __slots__ = ('max_temperature',)

    def __init__(self, temperature, max_temperature):
        self.max_temperature = float(max_temperature)
        super().__init__(temperature)

    @T08Sensor.temperature.setter
    def temperature(self, value):
        if value > self.max_temperature:
            raise ValueError('above device maximum')
        T08Sensor.temperature.fset(self, value)


sensor_08 = T08LimitedSensor(20, 80)
print(sensor_08.temperature, sensor_08.max_temperature)

20.0 80.0


**What just happened?** `@T08Sensor.temperature.setter` produces a child property reusing the inherited getter, with a new setter. The setter calls the base setter to reuse base validation. The child defines only `max_temperature` as a new slot.

**Solution · Step 3.** Check the child-specific restriction and the inherited restriction.

In [32]:
raises(ValueError, setattr, sensor_08, 'temperature', 81)
raises(TypeError, setattr, sensor_08, 'temperature', 'warm')
sensor_08.temperature = 79
print('Accepted:', sensor_08.temperature)

Accepted: 79.0


**Verification.** Prove the backing field remains a single inherited slot. These checks should pass when you run the notebook from top to bottom.

In [33]:
assert sensor_08.temperature == 79.0
assert sensor_08._temperature == 79.0
assert '_temperature' not in T08LimitedSensor.__dict__
assert not hasattr(sensor_08, '__dict__')

**Takeaway.** A property is not listed in `__slots__`; its backing attribute is. Subclasses can extend setter behavior while delegating to the parent’s property rather than duplicating parent storage.

**Try it yourself.** What happens when you assign `sensor_08.temperature = float("nan")`? Consider whether production validation needs a finite-number check.

## Problem 09 — Protect a required slot from ordinary deletion

**Challenge.** Prevent a client from deleting an initialized `record_id`, while allowing an optional `note` slot to be cleared. Explore the limit of this protection.

**Before running anything:** Does `__slots__` automatically make a field undeletable?

The next cells form a guided, fully worked solution. Read the explanation, make your prediction, then run the next cell.

**Solution · Step 1.** Implement a deletion guard on a base and declare an optional child slot.

In [34]:
class T09Document:
    __slots__ = ('record_id',)

    def __init__(self, record_id):
        self.record_id = record_id

    def __delattr__(self, name):
        if name == 'record_id':
            raise AttributeError('record_id is required')
        super().__delattr__(name)


class T09AnnotatedDocument(T09Document):
    __slots__ = ('note',)


doc_09 = T09AnnotatedDocument('D-001')
doc_09.note = 'reviewed'
print(doc_09.record_id, doc_09.note)

D-001 reviewed


**Solution · Step 2.** Delete the optional value, then try deleting the required value.

In [35]:
del doc_09.note
print('Optional note present?', hasattr(doc_09, 'note'))
raises(AttributeError, delattr, doc_09, 'record_id')
print('Still present:', doc_09.record_id)

Optional note present? False
Still present: D-001


**What just happened?** The override protects ordinary `del obj.record_id` syntax but does **not** make the object tamper-proof. Low-level calls such as `object.__delattr__` can bypass the custom override.

**Solution · Step 3.** Demonstrate the bypass on a disposable instance rather than damaging the primary example.

In [36]:
disposable_09 = T09Document('TEMP')
object.__delattr__(disposable_09, 'record_id')
print('Bypass removed storage:', not hasattr(disposable_09, 'record_id'))

Bypass removed storage: True


**Verification.** Normal deletion left the primary object valid and optional deletion succeeded. These checks should pass when you run the notebook from top to bottom.

In [37]:
assert doc_09.record_id == 'D-001'
assert not hasattr(doc_09, 'note')
assert not hasattr(disposable_09, 'record_id')

**Takeaway.** A slot can be deleted unless higher-level logic prevents it. Overriding deletion is a normal-API invariant, not strong immutability or a security mechanism.

**Try it yourself.** Give `note` an initial value in the child initializer and test deletion twice.

## Problem 10 — Represent a partially initialized slotted object safely

**Challenge.** Implement a `__repr__` that handles optional slots without crashing, and add a child representation including one more optional field.

**Before running anything:** Will `self.payload!r` work when `payload` has never been assigned?

The next cells form a guided, fully worked solution. Read the explanation, make your prediction, then run the next cell.

**Solution · Step 1.** Start with a base class that gives only the required identifier a value.

In [38]:
class T10Envelope:
    __slots__ = ('envelope_id', 'payload')

    def __init__(self, envelope_id):
        self.envelope_id = envelope_id

    def __repr__(self):
        payload = getattr(self, 'payload', MISSING)
        payload_text = '<unset>' if payload is MISSING else repr(payload)
        return f'T10Envelope(id={self.envelope_id!r}, payload={payload_text})'


empty_10 = T10Envelope('E-4')
print(repr(empty_10))

T10Envelope(id='E-4', payload=<unset>)


**What just happened?** A defensive representation must not assume that every declared slot is initialized. Using `repr(payload)` for initialized values makes quotes visible and distinguishes strings from other objects.

**Solution · Step 2.** Extend the representation through a child method, still handling an unset child field.

In [39]:
class T10TrackedEnvelope(T10Envelope):
    __slots__ = ('tracking_code',)

    def __repr__(self):
        code = getattr(self, 'tracking_code', MISSING)
        code_text = '<unset>' if code is MISSING else repr(code)
        return f'T10TrackedEnvelope({super().__repr__()}, tracking={code_text})'


tracked_10 = T10TrackedEnvelope('E-5')
tracked_10.payload = None
print(repr(tracked_10))

T10TrackedEnvelope(T10Envelope(id='E-5', payload=None), tracking=<unset>)


**Solution · Step 3.** Initialize the remaining field and examine the new representation.

In [40]:
tracked_10.tracking_code = 'ZX-1'
print(repr(tracked_10))

T10TrackedEnvelope(T10Envelope(id='E-5', payload=None), tracking='ZX-1')


**Verification.** Confirm that `None` and an unset slot have different representations. These checks should pass when you run the notebook from top to bottom.

In [41]:
assert 'payload=<unset>' in repr(empty_10)
assert 'payload=None' in repr(tracked_10)
assert "tracking='ZX-1'" in repr(tracked_10)
assert not hasattr(tracked_10, '__dict__')

**Takeaway.** Useful `__repr__` implementations should make debugging safer, not fail when the object is in a legitimate intermediate state. Use explicit optional-field handling when slots can be unset.

**Try it yourself.** Modify the child repr to display its public class name dynamically, without changing stored fields.

## Problem 11 — Equality and hashing: slots do not choose value semantics

**Challenge.** Create a value-like slotted base and child. Define equality deliberately; keep the objects unhashable because the fields are mutable.

**Before running anything:** If two slotted instances contain the same values, are they automatically equal? Are they automatically safe dictionary keys?

The next cells form a guided, fully worked solution. Read the explanation, make your prediction, then run the next cell.

**Solution · Step 1.** Compare two ordinary slotted objects before providing `__eq__`.

In [42]:
class T11Default:
    __slots__ = ('key',)

    def __init__(self, key):
        self.key = key


first_11 = T11Default('A')
second_11 = T11Default('A')
print('Same value?', first_11.key == second_11.key)
print('Equal objects by default?', first_11 == second_11)

Same value? True
Equal objects by default? False


**What just happened?** Slots only influence attribute layout. With no custom equality, ordinary instances use identity-based equality. Adding value-based equality requires deciding which fields participate and how subclasses compare.

**Solution · Step 2.** Create an explicit equality contract using exact types and a tuple of public state.

In [43]:
class T11Key:
    __slots__ = ('key',)

    def __init__(self, key):
        self.key = key

    def _identity_fields(self):
        return (self.key,)

    def __eq__(self, other):
        if type(self) is not type(other):
            return NotImplemented
        return self._identity_fields() == other._identity_fields()

    __hash__ = None  # Document the mutability decision explicitly.


class T11VersionedKey(T11Key):
    __slots__ = ('version',)

    def __init__(self, key, version):
        super().__init__(key)
        self.version = version

    def _identity_fields(self):
        return (*super()._identity_fields(), self.version)

    __hash__ = None


key_a_11 = T11VersionedKey('A', 1)
key_b_11 = T11VersionedKey('A', 1)
key_c_11 = T11VersionedKey('A', 2)
print(key_a_11 == key_b_11, key_a_11 == key_c_11)

True False


**Solution · Step 3.** Show why mutable value-based objects should not receive a naive hash implementation.

In [44]:
print('Cross-class equality:', T11Key('A') == key_a_11)
raises(TypeError, hash, key_a_11)
key_b_11.version = 3
print('Equality after mutation:', key_a_11 == key_b_11)

Cross-class equality: False
Equality after mutation: False


**Verification.** Confirm exact-class equality and the explicit no-hash policy. These checks should pass when you run the notebook from top to bottom.

In [45]:
assert T11VersionedKey('A', 1) == T11VersionedKey('A', 1)
assert T11VersionedKey('A', 1) != T11VersionedKey('A', 2)
assert T11VersionedKey('A', 1) != T11Key('A')
assert T11VersionedKey.__hash__ is None

**Takeaway.** Slots do not grant immutability, equality, or safe hashing. For mutable value-based objects, `__hash__ = None` avoids a dictionary-key contract that mutation could break.

**Try it yourself.** Design a separate immutable public API and state which low-level operations would still bypass it.

## Problem 12 — Pattern matching needs an explicit positional interface

**Challenge.** Make a slotted base and child work with Python structural pattern matching, first by keyword and then by positional patterns.

**Before running anything:** Does Python derive `__match_args__` automatically from ordinary `__slots__` definitions?

The next cells form a guided, fully worked solution. Read the explanation, make your prediction, then run the next cell.

**Solution · Step 1.** Construct classes that offer named fields but no positional matching contract.

In [46]:
class T12Event:
    __slots__ = ('event_id',)

    def __init__(self, event_id):
        self.event_id = event_id


class T12Alert(T12Event):
    __slots__ = ('severity',)

    def __init__(self, event_id, severity):
        super().__init__(event_id)
        self.severity = severity


alert_12 = T12Alert('EV-9', 3)
match alert_12:
    case T12Alert(event_id=eid, severity=level):
        print('Keyword pattern:', eid, level)

Keyword pattern: EV-9 3


**What just happened?** Keyword class patterns look up named attributes; slots support them naturally. Positional patterns use `__match_args__`, which an ordinary slotted class does **not** obtain automatically.

**Solution · Step 2.** Try a positional pattern via a small function so we can trap the exception safely.

In [47]:
def positional_match_without_contract_12(obj):
    match obj:
        case T12Alert(eid, level):
            return eid, level
    return None


raises(TypeError, positional_match_without_contract_12, alert_12)
print('No positional contract on T12Alert.')

No positional contract on T12Alert.


**Solution · Step 3.** Implement explicit positional contracts, including the inherited field in the child.

In [48]:
class T12PatternEvent:
    __slots__ = ('event_id',)
    __match_args__ = ('event_id',)

    def __init__(self, event_id):
        self.event_id = event_id


class T12PatternAlert(T12PatternEvent):
    __slots__ = ('severity',)
    __match_args__ = ('event_id', 'severity')

    def __init__(self, event_id, severity):
        super().__init__(event_id)
        self.severity = severity


pattern_alert_12 = T12PatternAlert('EV-10', 5)
match pattern_alert_12:
    case T12PatternAlert(eid, 5):
        matched_12 = eid
    case _:
        matched_12 = None
print('Positional match result:', matched_12)

Positional match result: EV-10


**Verification.** Test both the public pattern contract and the storage layout. These checks should pass when you run the notebook from top to bottom.

In [49]:
assert matched_12 == 'EV-10'
assert T12PatternAlert.__match_args__ == ('event_id', 'severity')
assert T12PatternAlert.__slots__ == ('severity',)
assert not hasattr(pattern_alert_12, '__dict__')

**Takeaway.** `__slots__` declares storage; `__match_args__` declares positional pattern-matching API. Do not conflate them. Explicit `__match_args__` can be a useful, intentional public contract.

**Try it yourself.** Reorder `__match_args__` and predict how an existing positional pattern changes meaning.

## Problem 13 — Why `cached_property` works on one child but fails on its base

**Challenge.** Place a `cached_property` on a dictionary-free base, demonstrate its failure, then observe an unslotted child that can cache it. Finally, build an entirely slotted alternative.

**Before running anything:** Can `cached_property` store its result in a regular slotted instance with no `__dict__`?

The next cells form a guided, fully worked solution. Read the explanation, make your prediction, then run the next cell.

**Solution · Step 1.** Define a cached computation inherited by both kinds of instance.

In [50]:
class T13Base:
    __slots__ = ('input_value',)

    def __init__(self, input_value):
        self.input_value = input_value

    @cached_property
    def doubled(self):
        return self.input_value * 2


class T13OpenChild(T13Base):
    pass  # Inherits input_value slot, but also receives an instance dictionary.


base_13 = T13Base(7)
child_13 = T13OpenChild(7)
raises(TypeError, getattr, base_13, 'doubled')
print('Unslotted child first read:', child_13.doubled)
print('Cached in child dictionary:', child_13.__dict__)

Unslotted child first read: 14
Cached in child dictionary: {'doubled': 14}


**What just happened?** `functools.cached_property` expects a mutable instance `__dict__` where it writes the cached result. The base lacks one; its unslotted child has one. The result is stored under `doubled` in the child dictionary, separately from `input_value` in the inherited slot.

**Solution · Step 2.** Replace the standard decorator with an explicit slotted cache and a property.

In [51]:
class T13ClosedCache:
    __slots__ = ('input_value', '_cached_double')

    def __init__(self, input_value):
        self.input_value = input_value
        self._cached_double = MISSING

    @property
    def doubled(self):
        if self._cached_double is MISSING:
            self._cached_double = self.input_value * 2
        return self._cached_double


class T13ClosedChild(T13ClosedCache):
    __slots__ = ()


closed_13 = T13ClosedChild(7)
print('Custom cache:', closed_13.doubled)
print('Read again:', closed_13.doubled)

Custom cache: 14
Read again: 14


**Solution · Step 3.** Inspect the storage differences and avoid promising that caching always stays valid when inputs change.

In [52]:
print('Open child dictionary:', child_13.__dict__)
print('Closed child dictionary?', hasattr(closed_13, '__dict__'))
print('Explicit cached slot:', closed_13._cached_double)

Open child dictionary: {'doubled': 14}
Closed child dictionary? False
Explicit cached slot: 14


**Verification.** Confirm the two valid caching approaches. These checks should pass when you run the notebook from top to bottom.

In [53]:
assert child_13.doubled == 14
assert child_13.__dict__ == {'doubled': 14}
assert closed_13.doubled == 14
assert not hasattr(closed_13, '__dict__')

**Takeaway.** A `cached_property` needs writable per-instance dictionary storage. For fully slotted objects, use an explicit cache slot and a regular property, then plan invalidation when inputs can change.

**Try it yourself.** Set `closed_13.input_value = 8`; explain why the cache is now stale. The next problem implements a remedy.

## Problem 14 — Cache invalidation across a slotted subclass

**Challenge.** Give a base class a validated, changeable score. Let its slotted subclass cache a derived classification, and invalidate that cache whenever the score changes.

**Before running anything:** Should the base class know the child’s cache-slot name?

The next cells form a guided, fully worked solution. Read the explanation, make your prediction, then run the next cell.

**Solution · Step 1.** Create a reusable base property and a no-op hook for changes.

In [54]:
class T14Score:
    __slots__ = ('_score',)

    def __init__(self, score):
        self.score = score

    @property
    def score(self):
        return self._score

    @score.setter
    def score(self, value):
        if type(value) is not int or not 0 <= value <= 100:
            raise ValueError('score must be an integer from 0 to 100')
        self._score = value
        self._after_score_change()

    def _after_score_change(self):
        pass  # Extension hook: base does not assume any child cache exists.

**What just happened?** The base owns score validation and change notification. The child owns any derived state. Calling a subclass hook during base initialization is safe here because the child’s hook only assigns a declared slot; in general, be careful when calling overridable methods in constructors.

**Solution · Step 2.** Implement the cache and invalidate it in the child hook.

In [55]:
class T14Report(T14Score):
    __slots__ = ('_band_cache', 'computations')

    def __init__(self, score):
        self._band_cache = MISSING
        self.computations = 0
        super().__init__(score)

    def _after_score_change(self):
        self._band_cache = MISSING

    @property
    def band(self):
        if self._band_cache is MISSING:
            self.computations += 1
            self._band_cache = 'pass' if self.score >= 60 else 'review'
        return self._band_cache


report_14 = T14Report(72)
print(report_14.band, report_14.band, report_14.computations)

pass pass 1


**Solution · Step 3.** Modify the input and verify that a new computation occurs only on the next read.

In [56]:
report_14.score = 42
print('After change, before reading:', report_14.computations)
print('Recomputed:', report_14.band)
print('Total computations:', report_14.computations)
raises(ValueError, setattr, report_14, 'score', 102)

After change, before reading: 1
Recomputed: review
Total computations: 2


ValueError('score must be an integer from 0 to 100')

**Verification.** The cached result changes exactly once after a valid update; invalid updates preserve it. These checks should pass when you run the notebook from top to bottom.

In [57]:
assert report_14.band == 'review'
assert report_14.computations == 2
assert report_14.score == 42
assert not hasattr(report_14, '__dict__')

**Takeaway.** Keep validation and change hooks in the base; keep subclass cache names in the subclass. An explicit invalidation strategy matters more than merely having a cache slot.

**Try it yourself.** Add a `grade_letter` property with its own cache slot, and invalidate it from the same hook.

## Problem 15 — Reject an unslotted subclass at definition time

**Challenge.** Write a base class that enforces a dictionary-free single-inheritance policy. Reject child classes that forget `__slots__`, and reject duplicate inherited slot names.

**Before running anything:** At what time should a missing `__slots__` declaration fail: object creation or subclass definition?

The next cells form a guided, fully worked solution. Read the explanation, make your prediction, then run the next cell.

**Solution · Step 1.** Define an `__init_subclass__` rule. Check the child declaration before instances ever exist.

In [58]:
class T15ClosedBase:
    __slots__ = ('uid',)

    def __init__(self, uid):
        self.uid = uid

    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        if '__slots__' not in cls.__dict__:
            raise TypeError(f'{cls.__name__} must declare __slots__')
        declared = cls.__dict__['__slots__']
        if isinstance(declared, str):
            declared = (declared,)
        else:
            declared = tuple(declared)
        ancestor_names = set()
        for ancestor in cls.__mro__[1:]:
            slots = ancestor.__dict__.get('__slots__', ())
            ancestor_names.update((slots,) if isinstance(slots, str) else slots)
        repeats = set(declared) & ancestor_names
        if repeats:
            raise TypeError(f'redeclared inherited slots: {sorted(repeats)}')

**What just happened?** `__init_subclass__` runs when a subclass is created. The policy checks only ordinary single-inheritance descendants here; this example intentionally does not attempt to solve unrelated multiple-inheritance layout restrictions or hostile metaprogramming.

**Solution · Step 2.** Define the valid closed child and prove an accidentally open child is rejected.

In [59]:
class T15GoodChild(T15ClosedBase):
    __slots__ = ('level',)

    def __init__(self, uid, level):
        super().__init__(uid)
        self.level = level


def define_open_child_15():
    class OpenChild(T15ClosedBase):
        pass
    return OpenChild


raises(TypeError, define_open_child_15)
print('Missing-slots subclass rejected at class creation.')

Missing-slots subclass rejected at class creation.


**Solution · Step 3.** Reject a duplicate inherited slot but permit a deliberately empty child declaration.

In [60]:
def define_repeated_slot_15():
    class BadChild(T15ClosedBase):
        __slots__ = ('uid',)
    return BadChild


raises(TypeError, define_repeated_slot_15)

class T15NoNewFields(T15ClosedBase):
    __slots__ = ()

print('Empty declaration accepted:', T15NoNewFields.__slots__)

Empty declaration accepted: ()


**Verification.** Check instance behavior and the class-creation policy. These checks should pass when you run the notebook from top to bottom.

In [61]:
good_15 = T15GoodChild('U-3', 5)
assert (good_15.uid, good_15.level) == ('U-3', 5)
assert not hasattr(good_15, '__dict__')
assert not hasattr(T15NoNewFields('U-4'), '__dict__')

**Takeaway.** Use `__init_subclass__` to catch mistakes at class definition time. For a controlled project, enforcing an explicit slot declaration can prevent a future child from silently acquiring a dictionary.

**Try it yourself.** Enhance the policy to reject an explicit `"__dict__"` in declared slots. Explain why some applications might intentionally allow it instead.

## Problem 16 — Create a slotted child dynamically using `type`

**Challenge.** Generate a subclass at runtime with exactly two new slots. Show that class layout must be defined when the class is created, not patched onto existing instances.

**Before running anything:** Does adding a name to a class’s `__slots__` tuple afterward install a new slot descriptor?

The next cells form a guided, fully worked solution. Read the explanation, make your prediction, then run the next cell.

**Solution · Step 1.** Create a normal parent and dynamically construct a single-inheritance child.

In [62]:
class T16Base:
    __slots__ = ('base_id',)

    def __init__(self, base_id):
        self.base_id = base_id


T16DynamicChild = type(
    'T16DynamicChild',
    (T16Base,),
    {'__slots__': ('region', 'priority')},
)

dynamic_16 = T16DynamicChild('B-1')
dynamic_16.region = 'north'
dynamic_16.priority = 2
print(dynamic_16.base_id, dynamic_16.region, dynamic_16.priority)

B-1 north 2


**What just happened?** The `type(name, bases, namespace)` constructor executes the same basic class-creation machinery as a `class` statement. Defining `__slots__` in the namespace is what creates its descriptors.

**Solution · Step 2.** Inspect the installed descriptor names before mutating any declarations.

In [63]:
original_slots_16 = T16DynamicChild.__slots__
print('Declared:', original_slots_16)
print('Descriptor for region?', 'region' in T16DynamicChild.__dict__)
print('Descriptor for zone?', 'zone' in T16DynamicChild.__dict__)

Declared: ('region', 'priority')
Descriptor for region? True
Descriptor for zone? False


**Solution · Step 3.** Temporarily edit the class’s declarative metadata, then restore it. The layout remains unchanged.

In [64]:
try:
    T16DynamicChild.__slots__ = original_slots_16 + ('zone',)
    print('Modified metadata:', T16DynamicChild.__slots__)
    raises(AttributeError, setattr, dynamic_16, 'zone', 'east')
    print('New descriptor installed?', 'zone' in T16DynamicChild.__dict__)
finally:
    T16DynamicChild.__slots__ = original_slots_16

Modified metadata: ('region', 'priority', 'zone')
New descriptor installed? False


**Verification.** Confirm the dynamic child retained its original storage layout. These checks should pass when you run the notebook from top to bottom.

In [65]:
assert T16DynamicChild.__slots__ == ('region', 'priority')
assert (dynamic_16.base_id, dynamic_16.region, dynamic_16.priority) == (
    'B-1', 'north', 2
)
assert not hasattr(dynamic_16, '__dict__')

**Takeaway.** Slots are created during class construction. Later reassigning the class’s `__slots__` attribute changes metadata, not the instance memory layout; avoid doing so.

**Try it yourself.** Build a second dynamic subclass with a `zone` slot using `type`, rather than modifying the existing one.

## Problem 17 — Dictionary keys cannot override an inherited slot descriptor

**Challenge.** Use a child that intentionally has both an inherited slot and its own dictionary. Insert a duplicate name directly into the dictionary and investigate normal lookup and assignment.

**Before running anything:** If `obj.__dict__["code"]` disagrees with `obj.code`, which value wins?

The next cells form a guided, fully worked solution. Read the explanation, make your prediction, then run the next cell.

**Solution · Step 1.** Create a slotted base and a child that explicitly allows extra dictionary fields.

In [66]:
class T17Base:
    __slots__ = ('code',)

    def __init__(self, code):
        self.code = code


class T17Flexible(T17Base):
    __slots__ = ('__dict__',)


flex_17 = T17Flexible('SLOT')
flex_17.comment = 'review'
print('Slot:', flex_17.code)
print('Dictionary:', flex_17.__dict__)

Slot: SLOT
Dictionary: {'comment': 'review'}


**What just happened?** The instance has two distinct storage locations: inherited slot `code`, and a dictionary for extra names. A slot descriptor is a **data descriptor**, so it takes precedence over a same-named entry in the instance dictionary.

**Solution · Step 2.** Deliberately inject a conflicting dictionary key, then perform an ordinary attribute read.

In [67]:
flex_17.__dict__['code'] = 'DICT'
print('Dictionary code:', flex_17.__dict__['code'])
print('Normal code:    ', flex_17.code)

Dictionary code: DICT
Normal code:     SLOT


**Solution · Step 3.** Assign to the attribute normally and inspect both independent storage locations.

In [68]:
flex_17.code = 'UPDATED SLOT'
print('Normal code after assignment:', flex_17.code)
print('Dictionary code after assignment:', flex_17.__dict__['code'])

Normal code after assignment: UPDATED SLOT
Dictionary code after assignment: DICT


**Verification.** Confirm descriptor priority even when the dictionary has a duplicate key. These checks should pass when you run the notebook from top to bottom.

In [69]:
assert flex_17.code == 'UPDATED SLOT'
assert flex_17.__dict__['code'] == 'DICT'
assert flex_17.__dict__['comment'] == 'review'
assert T17Base.__dict__['code'].__get__(flex_17, T17Flexible) == 'UPDATED SLOT'

**Takeaway.** A manually injected dictionary key does not replace a data-descriptor-backed slot on ordinary access. Avoid storing conflicting names in the dictionary; it makes debugging and serialization confusing.

**Try it yourself.** Delete `flex_17.__dict__["code"]` and confirm normal lookup is unchanged.

## Problem 18 — Build a descriptor that stores values in a slot

**Challenge.** Implement a reusable validating descriptor assigned to the public name `voltage`. Store its actual instance value in `_voltage`, a slot declared by the base. Extend the base with one child slot.

**Before running anything:** Which names belong in `__slots__`: `voltage`, `_voltage`, both, or neither?

The next cells form a guided, fully worked solution. Read the explanation, make your prediction, then run the next cell.

**Solution · Step 1.** Implement a minimal descriptor with a class-level storage-name contract.

In [70]:
class T18Range:
    def __init__(self, storage_name, low, high):
        self.storage_name = storage_name
        self.low = low
        self.high = high

    def __set_name__(self, owner, name):
        self.public_name = name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return getattr(instance, self.storage_name)

    def __set__(self, instance, value):
        if type(value) not in (int, float) or not self.low <= value <= self.high:
            raise ValueError(f'{self.public_name} must be within [{self.low}, {self.high}]')
        setattr(instance, self.storage_name, float(value))

**What just happened?** The descriptor lives on the **class** under `voltage`. Its stored value lives in each instance under `_voltage`. Defining both as slot names would collide with the descriptor; defining neither would leave no per-instance storage in a dictionary-free class.

**Solution · Step 2.** Attach the descriptor to a slotted parent and add one slot in the child.

In [71]:
class T18Circuit:
    __slots__ = ('_voltage',)
    voltage = T18Range('_voltage', 0, 240)

    def __init__(self, voltage):
        self.voltage = voltage


class T18MonitoredCircuit(T18Circuit):
    __slots__ = ('channel',)

    def __init__(self, voltage, channel):
        super().__init__(voltage)
        self.channel = channel


circuit_18 = T18MonitoredCircuit(120, 'A')
print(circuit_18.voltage, circuit_18._voltage, circuit_18.channel)

120.0 120.0 A


**Solution · Step 3.** Try invalid writes and confirm the custom descriptor remains discoverable on the parent.

In [72]:
raises(ValueError, setattr, circuit_18, 'voltage', 241)
raises(ValueError, setattr, circuit_18, 'voltage', -1)
raises(ValueError, setattr, circuit_18, 'voltage', True)
print('Descriptor:', type(T18Circuit.voltage).__name__)
print('Valid stored value:', circuit_18.voltage)

Descriptor: T18Range
Valid stored value: 120.0


**Verification.** Check the single backing slot and the child’s independent slot. These checks should pass when you run the notebook from top to bottom.

In [73]:
assert circuit_18.voltage == 120.0
assert T18Circuit.__slots__ == ('_voltage',)
assert T18MonitoredCircuit.__slots__ == ('channel',)
assert 'voltage' in T18Circuit.__dict__
assert not hasattr(circuit_18, '__dict__')

**Takeaway.** Custom data descriptors and properties can validate a public attribute while a differently named slot stores the value. This is the descriptor relationship previewed by the source lesson, developed into a reusable validator.

**Try it yourself.** Add a new descriptor named `current`, backed by `_current`. Which class must own the additional storage slot?

## Problem 19 — Annotations are documentation, not storage slots

**Challenge.** Compare type annotations, class attributes, and actual slot descriptors across a base and child. Implement initialization that matches the declared layout.

**Before running anything:** Does an annotation like `price: float` automatically make `price` available as a slot?

The next cells form a guided, fully worked solution. Read the explanation, make your prediction, then run the next cell.

**Solution · Step 1.** Write a class with one actual slot and a second name that is annotation-only.

In [74]:
class T19Draft:
    __slots__ = ('sku',)
    sku: str
    price: float

    def __init__(self, sku):
        self.sku = sku


draft_19 = T19Draft('SKU-1')
print('Annotations:', T19Draft.__annotations__)
print('Actual slot descriptors:', [k for k in ('sku', 'price')
                                   if k in T19Draft.__dict__])
raises(AttributeError, setattr, draft_19, 'price', 19.99)

Annotations: {'sku': <class 'str'>, 'price': <class 'float'>}
Actual slot descriptors: ['sku']


AttributeError("'T19Draft' object has no attribute 'price' and no __dict__ for setting new attributes")

**What just happened?** `__annotations__` describes intended fields for tooling and readers; it does not create slot storage or runtime type enforcement. Here only `sku` was declared in `__slots__`.

**Solution · Step 2.** Design a child that intentionally adds the second field as its own slot.

In [75]:
class T19Product(T19Draft):
    __slots__ = ('price',)
    price: float

    def __init__(self, sku, price):
        super().__init__(sku)
        self.price = float(price)


product_19 = T19Product('SKU-2', 19.99)
print(product_19.sku, product_19.price)

SKU-2 19.99


**Solution · Step 3.** Distinguish annotation, a class-level constant, and per-instance storage.

In [76]:
print('Base annotation:', T19Draft.__annotations__['price'])
print('Child slot descriptor:', type(T19Product.__dict__['price']).__name__)
print('Child declared slots:', T19Product.__slots__)
raises(AttributeError, setattr, product_19, 'discount', 5)

Base annotation: <class 'float'>
Child slot descriptor: member_descriptor
Child declared slots: ('price',)


AttributeError("'T19Product' object has no attribute 'discount' and no __dict__ for setting new attributes")

**Verification.** Check actual storage exists only where it was explicitly declared. These checks should pass when you run the notebook from top to bottom.

In [77]:
assert product_19.price == 19.99
assert not hasattr(product_19, '__dict__')
assert 'price' not in T19Draft.__dict__
assert 'price' in T19Product.__dict__

**Takeaway.** Annotations document expected types; `__slots__` declares permitted storage names; explicit validation enforces runtime constraints. These mechanisms are independent.

**Try it yourself.** Change the constructor to reject a negative price rather than silently storing it.

## Problem 20 — Inspect without triggering a costly property

**Challenge.** Write a debugger utility that tells you whether an attribute is a slot descriptor or a property without executing a property getter that increments a counter.

**Before running anything:** Will normal `getattr(obj, "summary")` call the property? Does `inspect.getattr_static` behave differently?

The next cells form a guided, fully worked solution. Read the explanation, make your prediction, then run the next cell.

**Solution · Step 1.** Create a property with an observable side effect and one inherited slot.

In [78]:
import inspect


class T20Base:
    __slots__ = ('data', '_reads')

    def __init__(self, data):
        self.data = data
        self._reads = 0

    @property
    def summary(self):
        self._reads += 1
        return self.data.upper()


class T20Child(T20Base):
    __slots__ = ('tag',)


inspected_20 = T20Child('hello')
inspected_20.tag = 'demo'
print('Getter executions before inspection:', inspected_20._reads)

Getter executions before inspection: 0


**What just happened?** Ordinary `getattr` performs descriptor invocation, which can execute code, trigger database access, or change state. Introspection sometimes needs a way to see the raw descriptor instead.

**Solution · Step 2.** Use `inspect.getattr_static` to inspect without invoking the descriptor.

In [79]:
static_summary_20 = inspect.getattr_static(inspected_20, 'summary')
static_data_20 = inspect.getattr_static(inspected_20, 'data')
print('summary descriptor type:', type(static_summary_20).__name__)
print('data descriptor type:', type(static_data_20).__name__)
print('Getter executions after static inspection:', inspected_20._reads)

summary descriptor type: property
data descriptor type: member_descriptor
Getter executions after static inspection: 0


**Solution · Step 3.** Contrast with a normal property read and a normal slot read.

In [80]:
print('Normal property read:', inspected_20.summary)
print('Getter executions after normal read:', inspected_20._reads)
print('Normal inherited slot read:', inspected_20.data)

Normal property read: HELLO
Getter executions after normal read: 1
Normal inherited slot read: hello


**Verification.** Prove static inspection obtained the descriptors and triggered no extra getter calls. These checks should pass when you run the notebook from top to bottom.

In [81]:
assert isinstance(static_summary_20, property)
assert static_data_20 is T20Base.__dict__['data']
assert inspected_20._reads == 1
assert inspected_20.summary == 'HELLO'
assert inspected_20._reads == 2

**Takeaway.** Use `inspect.getattr_static` when debugging needs raw descriptors rather than the values returned by their `__get__` methods. Static inspection is an inspection aid, not a universal substitute for attribute access.

**Try it yourself.** Use `inspect.getattr_static(inspected_20, "tag")` and explain why it returns a descriptor rather than `"demo"`.

## Problem 21 — Capstone: evolve a closed class hierarchy while preserving an external API

**Challenge.** Refactor a conventional message type into a slotted base and two successive slotted subclasses. Retain validation, a derived property, a stable public record format, and round-trip construction.

**Before running anything:** Which state belongs in each class’s slots, and why should the record format not depend on `__dict__`?

The next cells form a guided, fully worked solution. Read the explanation, make your prediction, then run the next cell.

**Solution · Step 1.** Define the storage and validation responsibilities in the base.

In [82]:
class T21Message:
    __slots__ = ('_message_id', 'text')

    def __init__(self, message_id, text):
        if not isinstance(message_id, str) or not message_id.strip():
            raise ValueError('message_id must be nonempty text')
        if not isinstance(text, str):
            raise TypeError('text must be a string')
        self._message_id = message_id
        self.text = text

    @property
    def message_id(self):
        return self._message_id

    def to_record(self):
        return {'message_id': self.message_id, 'text': self.text}

**What just happened?** The property `message_id` is public; `_message_id` is its storage. `to_record` publishes an intentionally chosen schema rather than inspecting internal layout. There is no `__dict__` requirement.

**Solution · Step 2.** Add an audited message child with its own field and explicitly extend serialization.

In [83]:
class T21AuditedMessage(T21Message):
    __slots__ = ('reviewer',)

    def __init__(self, message_id, text, reviewer):
        super().__init__(message_id, text)
        if not isinstance(reviewer, str) or not reviewer.strip():
            raise ValueError('reviewer must be nonempty text')
        self.reviewer = reviewer

    def to_record(self):
        return {**super().to_record(), 'reviewer': self.reviewer}

**What just happened?** The child declares only its *new* field and delegates validation and serialization for inherited fields. This keeps one source of truth for the base contract.

**Solution · Step 3.** Extend again with one new slot plus a computed, read-only property.

In [84]:
class T21PriorityMessage(T21AuditedMessage):
    __slots__ = ('priority',)

    def __init__(self, message_id, text, reviewer, priority):
        super().__init__(message_id, text, reviewer)
        if type(priority) is not int or priority not in range(1, 6):
            raise ValueError('priority must be an integer from 1 through 5')
        self.priority = priority

    @property
    def is_urgent(self):
        return self.priority >= 4

    def to_record(self):
        return {**super().to_record(), 'priority': self.priority}

    @classmethod
    def from_record(cls, record):
        return cls(
            message_id=record['message_id'],
            text=record['text'],
            reviewer=record['reviewer'],
            priority=record['priority'],
        )


message_21 = T21PriorityMessage('M-15', 'Check batch', 'Rae', 5)
print(message_21.to_record())
print('Urgent:', message_21.is_urgent)

{'message_id': 'M-15', 'text': 'Check batch', 'reviewer': 'Rae', 'priority': 5}
Urgent: True


**Solution · Step 4.** Round-trip through a record, and separate public fields from computed and private fields.

In [85]:
record_21 = message_21.to_record()
restored_21 = T21PriorityMessage.from_record(record_21)
print('Same public record:', restored_21.to_record() == record_21)
print('Stored dictionary?', hasattr(restored_21, '__dict__'))
print('Computed property in serialized record?', 'is_urgent' in record_21)

Same public record: True
Stored dictionary? False
Computed property in serialized record? False


**Solution · Step 5.** Exercise failure cases and verify every class owns only its new fields.

In [86]:
raises(ValueError, T21PriorityMessage, 'M', 'text', 'Rae', 0)
raises(ValueError, T21PriorityMessage, 'M', 'text', '', 3)
raises(TypeError, T21PriorityMessage, 'M', None, 'Rae', 3)
print('Base slots: ', T21Message.__slots__)
print('Child slots:', T21AuditedMessage.__slots__)
print('Final slots:', T21PriorityMessage.__slots__)

Base slots:  ('_message_id', 'text')
Child slots: ('reviewer',)
Final slots: ('priority',)


**Verification.** Run the capstone contract end to end. These checks should pass when you run the notebook from top to bottom.

In [87]:
assert record_21 == {
    'message_id': 'M-15',
    'text': 'Check batch',
    'reviewer': 'Rae',
    'priority': 5,
}
assert restored_21.to_record() == record_21
assert restored_21.is_urgent
assert not hasattr(restored_21, '__dict__')
assert T21PriorityMessage.__slots__ == ('priority',)
raises(AttributeError, setattr, restored_21, 'unexpected', 123)
raises(AttributeError, setattr, restored_21, 'message_id', 'REASSIGNED')

AttributeError("property 'message_id' of 'T21PriorityMessage' object has no setter")

**Takeaway.** A well-designed closed hierarchy uses `super()` for inherited responsibilities, declares only new storage at each level, exposes properties instead of private fields, and defines a stable external schema explicitly. The read-only property prevents ordinary reassignment but should not be mistaken for absolute immutability.

**Try it yourself.** Add a fourth subclass `EscalatedMessage` with one new `escalation_reason` slot and extend `to_record` without rewriting any inherited fields.

---
## Synthesis: a mental model for future problems

The source lesson establishes the core rules: a slotted parent can have an unslotted child with a dictionary; explicitly slotted children preserve closed storage; a slotted child cannot erase an inherited dictionary; properties and slots are related through descriptors; and `"__dict__"` can deliberately enable mixed storage.

This **new tutorial** then built more advanced techniques on top of those rules. The recurring distinctions are:

- **Declaration vs. initialization:** a slot name can exist while its value is unset.
- **Declaration vs. metadata:** an iterator used for `__slots__` may be exhausted; changing the attribute later does not rebuild the layout.
- **Storage vs. API:** a slot descriptor, a property, a type annotation, a cached value, and a serialized record each serve a different purpose.
- **Parent vs. child responsibility:** the base owns its fields and validation; the child declares only its new fields and calls `super()`.
- **Convenience vs. guarantees:** slots are not runtime type checks, absolute immutability, security, or universal performance optimization.

**Notebook self-check:** Restart the kernel and run all cells. Every expected exception is captured, and every problem includes assertions. If you change a demonstration, rerun from the beginning because each problem’s assertions depend on its preceding guided steps.

## Final independent challenge (no provided implementation required)

Design a slotted `Task` with a validated, read-only public identifier; derive `TimedTask` with a strictly positive duration; then derive `CachedTask` with a cache for a formatted summary. Require: (1) no instance dictionaries anywhere; (2) only new slots in each child; (3) cache invalidation on duration updates; (4) a schema-based `to_record` method; and (5) at least eight assertions, including two invalid assignments. Use Problems 7, 8, 14, and 21 as patterns. Compare your planned slot declaration at each class level *before* implementing the code.

### Final independent challenge — complete reference solution

**Step 1: map responsibilities.** `Task` owns `_task_id` and exposes its read-only property. `TimedTask` adds `_duration` and the validated `duration` property. `CachedTask` adds `_summary_cache` and invalidates it through an overrideable hook. We deliberately initialize the cache **before** calling the superclass initializer, because the duration setter calls the hook during construction.

In [88]:
class FinalTask:
    __slots__ = ("_task_id",)

    def __init__(self, task_id):
        if not isinstance(task_id, str) or not task_id.strip():
            raise ValueError("task_id must be nonempty text")
        self._task_id = task_id

    @property
    def task_id(self):
        return self._task_id

    def to_record(self):
        return {"task_id": self.task_id}


class FinalTimedTask(FinalTask):
    __slots__ = ("_duration",)

    def __init__(self, task_id, duration):
        super().__init__(task_id)
        self.duration = duration

    @property
    def duration(self):
        return self._duration

    @duration.setter
    def duration(self, value):
        if type(value) not in (int, float) or not 0 < value < float("inf"):
            raise ValueError("duration must be a positive, finite number")
        self._duration = float(value)
        self._duration_changed()

    def _duration_changed(self):
        pass

    def to_record(self):
        return {**super().to_record(), "duration": self.duration}

**Step 2: add the cache without introducing an instance dictionary.** The subclass initializes its cache first, derives the summary on demand, and clears that cache when the parent property reports a duration change.

In [89]:
class FinalCachedTask(FinalTimedTask):
    __slots__ = ("_summary_cache",)

    def __init__(self, task_id, duration):
        self._summary_cache = MISSING
        super().__init__(task_id, duration)

    def _duration_changed(self):
        self._summary_cache = MISSING

    @property
    def summary(self):
        if self._summary_cache is MISSING:
            self._summary_cache = f"{self.task_id}: {self.duration:g} min"
        return self._summary_cache


final_task = FinalCachedTask("TK-1", 15)
print(final_task.summary)
final_task.duration = 20
print(final_task.summary)
print(final_task.to_record())

TK-1: 15 min
TK-1: 20 min
{'task_id': 'TK-1', 'duration': 20.0}


**Step 3: test the entire contract.** These assertions cover storage, serialization, update invalidation, public read-only behavior, and failure cases.

In [90]:
assert FinalTask.__slots__ == ("_task_id",)
assert FinalTimedTask.__slots__ == ("_duration",)
assert FinalCachedTask.__slots__ == ("_summary_cache",)
assert not hasattr(final_task, "__dict__")
assert final_task.summary == "TK-1: 20 min"
assert final_task.to_record() == {"task_id": "TK-1", "duration": 20.0}
raises(AttributeError, setattr, final_task, "task_id", "TK-2")
raises(AttributeError, setattr, final_task, "extra", 123)
raises(ValueError, setattr, final_task, "duration", 0)
raises(ValueError, setattr, final_task, "duration", -5)
raises(ValueError, setattr, final_task, "duration", float("inf"))
assert final_task.duration == 20.0
assert final_task.summary == "TK-1: 20 min"
print("Final independent challenge: all assertions passed.")

Final independent challenge: all assertions passed.


**Reference-solution discussion.** The getter for `task_id` has no ordinary setter, but low-level access to `_task_id` can still change storage: this is API-level read-only design, not a hard security guarantee. The duration check intentionally rejects boolean values and infinity; the comparison also rejects NaN. The explicit `to_record` contract keeps the public representation separate from both private slots and cache internals.